In [1]:
#Test project folder
import os, time
from pathlib import Path

PROJECT = Path("sample_project")
files = {                       # path: kitne din purani
    "main.py": 2,
    "utils/helpers.py": 10,
    "utils/old_helpers.py": 45,
    "docs/readme.md": 31,
    "docs/design.pdf": 120,
    "data/sales.csv": 400,
    "config/settings.json": 5,
    "LICENSE": 90,              # extension nahi
}

for rel, days_old in files.items():
    p = PROJECT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(f"sample content of {rel}\n" * 20)
    ts = time.time() - days_old * 86400
    os.utime(p, (ts, ts))       # modified time set karo

print("Sample project ready:", PROJECT.resolve())

Sample project ready: /content/sample_project


In [2]:
#Custom module
%%writefile dir_scanner.py
"""dir_scanner: project folder scan karne ka reusable module."""
import os
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

STALE_DAYS = 30          # itne din se purani file = stale
FINE_PER_DAY = 5.0       # har extra din ka fine
MAX_FINE_PER_FILE = 500.0


@dataclass
class FileRecord:
    name: str
    path: str
    size_bytes: int
    file_type: str
    modified: datetime
    days_old: int
    is_stale: bool
    fine: float


def get_file_type(path):
    ext = Path(path).suffix.lower().lstrip(".")
    return ext if ext else "no_extension"


def human_size(num_bytes):
    size = float(num_bytes)
    for unit in ("B", "KB", "MB", "GB"):
        if size < 1024:
            return f"{size:.1f} {unit}"
        size /= 1024
    return f"{size:.1f} TB"


def calculate_fine(days_old, stale_days=STALE_DAYS,
                   fine_per_day=FINE_PER_DAY, cap=MAX_FINE_PER_FILE):
    """30 din se oopar har din ka fine, cap tak."""
    overage = days_old - stale_days
    if overage <= 0:
        return 0.0
    return round(min(overage * fine_per_day, cap), 2)


def scan_directory(root, stale_days=STALE_DAYS, now=None):
    """Folder (subfolders samet) scan karo.
    Return: (records, errors). Ek kharab file poora scan nahi rokti."""
    root = Path(root)
    if not root.is_dir():
        raise NotADirectoryError(f"Folder nahi mila: {root}")

    now = now or datetime.now()
    records, errors = [], []

    for dirpath, _dirs, filenames in os.walk(root):
        for fname in filenames:
            p = Path(dirpath) / fname
            try:
                st = p.stat()
                modified = datetime.fromtimestamp(st.st_mtime)
                days_old = max((now - modified).days, 0)
                records.append(FileRecord(
                    name=fname,
                    path=str(p.relative_to(root)),
                    size_bytes=st.st_size,
                    file_type=get_file_type(fname),
                    modified=modified,
                    days_old=days_old,
                    is_stale=days_old > stale_days,
                    fine=calculate_fine(days_old, stale_days),
                ))
            except OSError as e:
                errors.append((str(p), str(e)))

    records.sort(key=lambda r: r.path)
    return records, errors

Writing dir_scanner.py


In [3]:
#Audit script (module import karke use karein)
import csv, importlib
from datetime import datetime
from pathlib import Path

importlib.invalidate_caches()          # notebook mein nayi file import hone ke liye
import dir_scanner as ds

TARGET_DIR = "sample_project"          # apna folder path yahan likhein
REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(exist_ok=True)

run_time = datetime.now()
stamp = run_time.strftime("%Y%m%d_%H%M%S")

records, errors = ds.scan_directory(TARGET_DIR, now=run_time)

# --- Audit report ---
audit_path = REPORT_DIR / f"audit_report_{stamp}.csv"
with open(audit_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["name", "path", "size", "type", "last_modified", "days_old", "status"])
    for r in records:
        w.writerow([r.name, r.path, ds.human_size(r.size_bytes), r.file_type,
                    r.modified.strftime("%Y-%m-%d %H:%M"), r.days_old,
                    "STALE" if r.is_stale else "OK"])

# --- Penalties report ---
penalised = [r for r in records if r.fine > 0]
total_fine = sum(r.fine for r in penalised)

penalty_path = REPORT_DIR / f"penalties_report_{stamp}.csv"
with open(penalty_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["path", "days_old", "overage_days", "fine"])
    for r in sorted(penalised, key=lambda x: x.fine, reverse=True):
        w.writerow([r.path, r.days_old, r.days_old - ds.STALE_DAYS, f"{r.fine:.2f}"])
    w.writerow(["TOTAL", "", "", f"{total_fine:.2f}"])

print("✅ Reports ban gayi:")
print("  ", audit_path)
print("  ", penalty_path)

✅ Reports ban gayi:
   reports/audit_report_20261023_112127.csv
   reports/penalties_report_20261023_112127.csv


In [4]:
#Neat summary
print("=" * 72)
print("PROJECT DIRECTORY AUDIT".center(72))
print(f"Scanned at: {run_time:%Y-%m-%d %H:%M:%S}  |  Folder: {TARGET_DIR}".center(72))
print("=" * 72)
print(f"{'File':<26}{'Type':<8}{'Size':>9}{'Modified':>14}{'Age':>6}  Status")
print("-" * 72)
for r in records:
    flag = "⚠️ STALE" if r.is_stale else "OK"
    print(f"{r.path:<26}{r.file_type:<8}{ds.human_size(r.size_bytes):>9}"
          f"{r.modified:%Y-%m-%d}{r.days_old:>6}d  {flag}")

print("\nPENALTIES")
print("-" * 72)
for r in sorted(penalised, key=lambda x: x.fine, reverse=True):
    print(f"  {r.path:<26} {r.days_old - ds.STALE_DAYS:>4} din overage   Rs. {r.fine:>8,.2f}")

print("=" * 72)
print(f"Total files: {len(records)} | Stale: {sum(r.is_stale for r in records)} "
      f"| Unreadable: {len(errors)} | Total fine: Rs. {total_fine:,.2f}")
print("=" * 72)

                        PROJECT DIRECTORY AUDIT                         
       Scanned at: 2026-10-23 11:21:27  |  Folder: sample_project       
File                      Type         Size      Modified   Age  Status
------------------------------------------------------------------------
LICENSE                   no_extension  520.0 B2026-07-25    90d  ⚠️ STALE
config/settings.json      json      780.0 B2026-10-18     5d  OK
data/sales.csv            csv       660.0 B2025-09-18   400d  ⚠️ STALE
docs/design.pdf           pdf       680.0 B2026-06-25   120d  ⚠️ STALE
docs/readme.md            md        660.0 B2026-09-22    31d  ⚠️ STALE
main.py                   py        520.0 B2026-10-21     2d  OK
utils/helpers.py          py        700.0 B2026-10-13    10d  OK
utils/old_helpers.py      py        780.0 B2026-09-08    45d  ⚠️ STALE

PENALTIES
------------------------------------------------------------------------
  data/sales.csv              370 din overage   Rs.   500.00
  docs/des